# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [ ]:
import os

# Specify GPU to use (e.g., GPU:0, CPU:-1)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Use CUDA async allocator to reduce fragmentation:
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# Suppress TensorFlow logging (0: ALL, 1: INFO, 2: WARNING, 3: ERROR):
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# # If it fails to determine best cudnn convolution algorithm
# os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

In [ ]:
# # Disable all auto-JIT clustering at the process level
# os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"

### 1.2. Imports

In [ ]:
from _imports import * # Centralized file containing all imports

### 1.3. GPU Management

In [ ]:
get_gpu_info()

## 2. Run Parameters 

In [ ]:
NUM_TRIALS = 1000
EPOCHS = 50

SAMPLER_SEED = 0

STEPS_PER_EXECUTION = 32

# Enable or disable XLA compilation
# Note: some layers don't support determinism with XLA
USE_JIT_COMPILE = True

In [ ]:
# Number of top trials to save
TOP_K = 3

# Order to rank trials by:
# "ascending" -> the lowest value is the best
# "descending" -> the highest value is the best
ORDER = "descending"

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "best_val_accuracy"

# Direction of optimization:
# "minimize" -> the lowest value is the best
# "maximize" -> the highest value is the best
DIRECTION = "minimize"


In [ ]:
# Label/target configuration.
# Soft labels keep the full beam-score distribution instead of argmax.
USE_SOFT_LABELS = True

# Label smoothing reduces overconfidence by mixing a uniform prior into targets.
LABEL_SMOOTHING = 0.05

# Focal loss focuses learning on harder examples.
USE_FOCAL_LOSS = True
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = None  # Set to a float (e.g., 0.25) to rebalance positives.

# Class weights compensate for skewed beam usage.
USE_CLASS_WEIGHTS = True
CLASS_WEIGHT_CLIP = (0.25, 4.0)
BEAM_SCORE_EPS = 1e-8


In [ ]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

BYTES_PER_PARAM = tf.dtypes.as_dtype(POLICY.variable_dtype).size

In [ ]:
# Set to an existing dir to resume training
RUN_DIR = f"runs/{get_caller_stem()}"  # (e.g. "runs/nas_1")

## 3. Data Loading and Preprocessing

In [ ]:
def normalize_beam_scores(beam_scores: np.ndarray, eps: float = BEAM_SCORE_EPS) -> np.ndarray:
    """Normalize per-sample beam scores into a probability distribution."""
    # Keeps relative confidence across beams instead of collapsing to argmax.
    flat = beam_scores.reshape(beam_scores.shape[0], -1).astype(np.float32)
    flat = np.maximum(flat, 0.0)
    denom = flat.sum(axis=1, keepdims=True)
    probs = flat / np.maximum(denom, eps)
    zero_rows = denom.squeeze() <= eps
    if np.any(zero_rows):
        probs[zero_rows] = 1.0 / probs.shape[1]
    return probs


def apply_label_smoothing(labels: np.ndarray, smoothing: float) -> np.ndarray:
    """Mix uniform probability into targets to reduce overconfidence."""
    if smoothing <= 0.0:
        return labels
    num_classes = labels.shape[1]
    return (1.0 - smoothing) * labels + (smoothing / num_classes)


def make_soft_labels(beam_scores: np.ndarray, label_smoothing: float = 0.0) -> np.ndarray:
    """Create soft labels from beam scores; optional smoothing regularizes targets."""
    probs = normalize_beam_scores(beam_scores)
    probs = apply_label_smoothing(probs, label_smoothing)
    return probs.astype(np.float32)


def compute_class_weights_from_soft_labels(
    labels: np.ndarray,
    clip: Tuple[float, float] = CLASS_WEIGHT_CLIP,
    eps: float = BEAM_SCORE_EPS,
) -> np.ndarray:
    """Compute inverse-frequency weights to upweight rare beams."""
    class_mass = labels.sum(axis=0)
    total = class_mass.sum()
    weights = total / (labels.shape[1] * np.maximum(class_mass, eps))
    weights = np.clip(weights, clip[0], clip[1])
    return (weights / np.mean(weights)).astype(np.float32)


def build_weighted_categorical_focal_loss(
    *,
    use_focal: bool,
    focal_gamma: float,
    focal_alpha: Optional[Union[float, np.ndarray]],
    class_weights: Optional[np.ndarray],
) -> Callable[[tf.Tensor, tf.Tensor], tf.Tensor]:
    """Return a loss that supports focal scaling and per-class weights."""
    class_weights_tf = None
    if class_weights is not None:
        class_weights_tf = tf.constant(class_weights, dtype=tf.float32)

    alpha_tf = None
    if focal_alpha is not None:
        alpha_tf = tf.constant(focal_alpha, dtype=tf.float32)

    def loss(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
        # Focal loss emphasizes hard examples; class weights rebalance skewed labels.
        y_true = tf.cast(y_true, y_pred.dtype)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())

        ce = -y_true * tf.math.log(y_pred)

        if class_weights_tf is not None:
            ce = ce * tf.cast(class_weights_tf, y_pred.dtype)
        if alpha_tf is not None:
            ce = ce * tf.cast(alpha_tf, y_pred.dtype)
        if use_focal and focal_gamma > 0:
            ce = ce * tf.pow(1.0 - y_pred, focal_gamma)

        return tf.reduce_sum(ce, axis=-1)

    return loss


def load_dataset_soft_labels(
    s008_path: str = "./data/s008",
    s009_path: str = "./data/s009",
    *,
    use_soft_labels: bool = True,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Load s008/s009 and return targets as soft distributions (no smoothing)."""
    # s008 train + val
    s008_y_train = np.load(os.path.join(s008_path, "beam_output", "beam_output_train.npz"))["output_classification"]
    s008_y_val = np.load(os.path.join(s008_path, "beam_output", "beam_output_val.npz"))["output_classification"]
    s008_coord_train = np.load(os.path.join(s008_path, "coord_input", "coord_train.npz"))["coordinates"]
    s008_coord_val = np.load(os.path.join(s008_path, "coord_input", "coord_val.npz"))["coordinates"]
    s008_lidar_train = np.load(os.path.join(s008_path, "lidar_input", "lidar_train.npz"))["input"]
    s008_lidar_val = np.load(os.path.join(s008_path, "lidar_input", "lidar_val.npz"))["input"]

    s008_y = np.concatenate((s008_y_train, s008_y_val), axis=0)
    s008_coord_input = np.concatenate((s008_coord_train, s008_coord_val), axis=0).astype(np.float32)
    s008_lidar_input = np.concatenate((s008_lidar_train, s008_lidar_val), axis=0)

    # s009 full
    s009_y = np.load(os.path.join(s009_path, "beam_output", "beam_output.npz"))["output_classification"]
    s009_coord_input = np.load(os.path.join(s009_path, "coord_input", "coord_input.npz"))["coordinates"].astype(np.float32)
    s009_lidar_input = np.load(os.path.join(s009_path, "lidar_input", "lidar_input.npz"))["input"]

    if use_soft_labels:
        # Soft labels keep beam-score distribution for richer supervision.
        s008_y = make_soft_labels(s008_y, label_smoothing=0.0)
        s009_y = make_soft_labels(s009_y, label_smoothing=0.0)
    else:
        # Hard labels (one-hot) for compatibility if needed.
        s008_y = tf.keras.utils.to_categorical(
            np.argmax(s008_y.reshape(s008_y.shape[0], -1), axis=1),
            num_classes=256,
        ).astype(np.float32)
        s009_y = tf.keras.utils.to_categorical(
            np.argmax(s009_y.reshape(s009_y.shape[0], -1), axis=1),
            num_classes=256,
        ).astype(np.float32)

    return (
        s008_coord_input,
        s008_lidar_input,
        s008_y,
        s009_coord_input,
        s009_lidar_input,
        s009_y,
    )


In [ ]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_soft_labels(
    s008_path="./data/s008",
    s009_path="./data/s009",
    use_soft_labels=USE_SOFT_LABELS,
)

## Hyperparameters

In [ ]:
kparams = KParams(
    activation_choices={
        "relu": tf.keras.activations.relu,
        "silu": tf.keras.activations.silu,
    },
    regularizer_choices={
        # "l2": tf.keras.regularizers.l2,
        "none": None,
    },
    optimizer_choices={
        # "sgd": tf.keras.optimizers.SGD(momentum=0.9),
        # "adam": tf.keras.optimizers.Adam(),
        "adamw": tf.keras.optimizers.AdamW(weight_decay=1e-4),
        # "lion": tf.keras.optimizers.Lion(beta_1=0.9, beta_2=0.99),
        # "rmsprop": tf.keras.optimizers.RMSprop(),
    },
    # scaler_choices={
    #     "standard": StandardScaler,
    #     "minmax_0_1": lambda: MinMaxScaler(feature_range=(0, 1)),
    #     "minmax_-1_1": lambda: MinMaxScaler(feature_range=(-1, 1)),
    # },
    learning_rate=(1e-4, 1e-2),
)

## 5. Model Definition

In [ ]:
def build_model(
    trial: optuna.Trial,
    kparams: dict,
    *,
    show_summary: bool = True,
    **kwargs: Any,
) -> tf.keras.Model:

    train_seed = kwargs.get("train_seed")
    loss_fn = kwargs.get("loss_fn")  # Inject custom loss (soft labels, focal, class weights).

    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=train_seed,
    )

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Optional 2D stem to keep spatial structure before flattening.
    use_conv2d_stem = trial.suggest_categorical("use_conv2d_stem", [True, False])
    lidar_channels = 4
    if use_conv2d_stem:
        stem_filters = trial.suggest_categorical("conv2d_stem_filters", [8, 16])
        stem_kernel = 3
        stem_act = trial.suggest_categorical("conv2d_stem_act", ["relu", "silu"])

        one_hot_lidar = layers.Conv2D(
            filters=stem_filters,
            kernel_size=(stem_kernel, stem_kernel),
            padding="same",
            activation=None,
            kernel_initializer=initializer,
            name="lidar_stem_conv2d",
        )(one_hot_lidar)
        one_hot_lidar = layers.BatchNormalization(name="lidar_stem_bn")(one_hot_lidar)
        one_hot_lidar = layers.Activation(stem_act, name="lidar_stem_act")(one_hot_lidar)
        lidar_channels = stem_filters

    # Downsample the grid before flattening to cut the sequence length (and FLOPs).
    one_hot_lidar = layers.MaxPooling2D(
        pool_size=(2, 2),
        name="lidar_pool_2d",
    )(one_hot_lidar)

    seq_len = (20 // 2) * (200 // 2)

    # Flatten the pooled grid into a shorter sequence with the channels.
    x_lidar_flat: layers.Layer = layers.Reshape(
        (seq_len, lidar_channels),
        name="lidar_flatten_channels",
    )(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,seq_len,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, seq_len, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(seq_len, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,seq_len,channels) + (batch,seq_len,2) → (batch,seq_len,channels+2)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————————— CNN ——————————————————————————————————— #
    num_conv_layers = trial.suggest_int("num_conv_layers", 2, 4)
    use_residual = trial.suggest_categorical("use_residual", [True, False])

    x = combined
    for i in range(num_conv_layers):
        filter_choices = [32, 64, 96, 128]
        filters = trial.suggest_categorical(f"conv1d_{i}_filters", filter_choices)
        x_in = x
        x = build_cnn1d(
            trial=trial,
            kparams=kparams,
            x=x,
            name_prefix=f"conv1d_{i}",
            # Filters
            filters_range=filters,
            # Kernel size
            kernel_size_range=(3, 7),
            kernel_size_step=2,
            kernel_initializer=initializer,
        )
        if use_residual:
            if x_in.shape[-1] != filters:
                x_in = layers.Conv1D(
                    filters=filters,
                    kernel_size=1,
                    padding="same",
                    kernel_initializer=initializer,
                    name=f"res_proj_{i}",
                )(x_in)
            x = layers.Add(name=f"res_add_{i}")([x_in, x])

        #! pool size = 1 means no downsampling
        pool_size = trial.suggest_int(f"pool_size_{i}", 1, 2, step=1)
        if pool_size > 1:
            x = layers.MaxPooling1D(pool_size=pool_size, name=f"max_pool_{i}")(x)

    pooling_type = trial.suggest_categorical("pooling_type", ["max", "average"])
    if pooling_type == "max":
        x = layers.GlobalMaxPooling1D(name="global_max_pooling")(x)
    else:
        x = layers.GlobalAveragePooling1D(name="global_avg_pooling")(x)

    num_dense_layers = trial.suggest_int("num_dense_layers", 0, 1)
    for i in range(num_dense_layers):
        x = build_dnn(
            trial=trial,
            kparams=kparams,
            x=x,
            name_prefix=f"dense_{i}",
            units_range=(32, 128),
            units_step=32,
            dropout_rate_range=(0.0, 0.3),
            dropout_rate_step=0.1,
            kernel_initializer=initializer,
        )

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    model.compile(
        optimizer=kparams.get_optimizer(trial),
        loss=loss_fn if loss_fn is not None else losses.CategoricalCrossentropy(),
        metrics=[metrics.CategoricalAccuracy(name="accuracy")],
        jit_compile=USE_JIT_COMPILE,  # For XLA speedup, does not support determinism
        steps_per_execution=STEPS_PER_EXECUTION,
    )

    return model

## 6. Objective Function

In [ ]:
def objective(
    trial: optuna.Trial,
    **kwargs: Any,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        **kwargs: Additional keyword arguments.

    Returns:
        float: Final value used for optimization.
    """
    (print(f"Running trial {trial.number}..."), clear_session())

    # —————————————————————————————— Reproducibility ————————————————————————————— #
    DATA_SEED = 0
    TRAIN_SEED = 0

    # Set Python, NumPy, Keras and TensorFlow seeds
    set_random_seed(TRAIN_SEED)

    # ——————————————————————————————————— Setup —————————————————————————————————— #
    global s009_coord_input, s009_lidar_input, s009_y
    global s008_coord_input, s008_lidar_input, s008_y_train

    (
        x_s008_lidar_train,
        x_s008_lidar_val,
        x_s008_coord_train,
        x_s008_coord_val,
        y_s008_train,
        y_s008_val,
    ) = train_test_split(
        s008_lidar_input,
        s008_coord_input,
        s008_y_train,
        test_size=0.2,
        random_state=DATA_SEED,
        shuffle=True,
    )

    # Class weights computed before smoothing to preserve true label frequencies.
    class_weights = None
    if USE_CLASS_WEIGHTS:
        class_weights = compute_class_weights_from_soft_labels(y_s008_train, clip=CLASS_WEIGHT_CLIP)

    # Apply label smoothing only to training targets to reduce overconfidence.
    if LABEL_SMOOTHING > 0.0:
        y_s008_train = apply_label_smoothing(y_s008_train, LABEL_SMOOTHING)

    backup_dir = kwargs["backup"]
    model_dir = kwargs["model"]
    fig_dir = kwargs["fig"]
    tensorboard_dir = kwargs["tensorboard"]
    logs_dir = kwargs["logs"]
    history_dir = kwargs["history"]
    scaler_dir = kwargs["scaler"]

    # ———————————————————————————————————————————————————————————————————————————— #

    try:
        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Data Preprocessing                              #
        # ———————————————————————————————————————————————————————————————————————————— #
        coord_scaler = StandardScaler()

        coord_scaler.fit(x_s008_coord_train)
        x_s008_coord_train = coord_scaler.transform(x_s008_coord_train)
        x_s008_coord_val = coord_scaler.transform(x_s008_coord_val)
        s009_coord_input = coord_scaler.transform(s009_coord_input)
        s008_coord_input = coord_scaler.transform(s008_coord_input)

        scaler_path = os.path.join(scaler_dir, f"trial_{trial.number}.pkl")
        with open(scaler_path, "wb") as scaler_file:
            pickle.dump(coord_scaler, scaler_file)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                        Model Construction and Training                       #
        # ———————————————————————————————————————————————————————————————————————————— #
        # Custom loss mixes focal scaling with class weights for imbalanced beams.
        loss_fn = build_weighted_categorical_focal_loss(
            use_focal=USE_FOCAL_LOSS,
            focal_gamma=FOCAL_GAMMA,
            focal_alpha=FOCAL_ALPHA,
            class_weights=class_weights,
        )

        model = build_model(
            trial=trial,
            kparams=kparams,
            show_summary=False,
            train_seed=TRAIN_SEED,
            loss_fn=loss_fn,
        )
        BATCH_SIZE = 64

        prune_model_by_config(
            trial=trial,
            model=model,
            thresholds={
                "model_size": 50,  # Maximum model size in MB
                "memory_mb": 9000,  # Maximum memory training usage in MB
                "param": 350000,  # Maximum number of parameters
                "flops": 5e8,  # Maximum number of FLOPs
            },
            bytes_per_param=BYTES_PER_PARAM,
            batch_size=BATCH_SIZE,
        )

        history = model.fit(
            x=[x_s008_lidar_train, x_s008_coord_train],
            y=y_s008_train,
            validation_data=([x_s008_lidar_val, x_s008_coord_val], y_s008_val),
            epochs=EPOCHS,
            batch_size=1,
            callbacks=get_callbacks_study(
                trial=trial,
                monitor="val_loss",
                #! Can cause high memory usage
                # tensorboard_logs=tensorboard_dir,
                reduce_lr_patience=3,
                early_stopping_patience=5,
            ),
            verbose=2,
        )

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Save Trial Results                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ——————————————————————————— Model characteristics —————————————————————————— #
        set_user_attr_model_stats(
            trial=trial,
            model=model,
            bytes_per_param=BYTES_PER_PARAM,
            batch_size=BATCH_SIZE,
            test_runs=10,
            device="gpu/0",
            stats_to_measure=(
                "parameters",
                "model_size",
                "flops",
                "macs",
                "summary",
                "inference_latency",
                # "cpu_util_percent",
                # "cpu_power_rapl_w",
                # "ram_used_bytes",
                # "ram_util_percent",
                # "gpu_util_percent",
                # "gpu_mem_used_bytes",
                # "gpu_power_w",
            ),
            extra_attrs=None,
            verbose=1,
        )

        # Evaluate on full s009
        s009_loss, s009_acc = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=BATCH_SIZE, verbose=2
        )

        # Evaluate on s008
        s008_loss, s008_acc = model.evaluate(
            [s008_lidar_input, s008_coord_input], s008_y_train, batch_size=BATCH_SIZE, verbose=2
        )

        # ———————————————————————————————————————————————————————————————————————————— #
        #                               Extra Attributes                               #
        # ———————————————————————————————————————————————————————————————————————————— #
        # # Choose best epoch based on validation loss
        # if "minimize" in DIRECTION:
        #     # The best epoch is the one with the lowest validation loss
        #     best_idx = int(np.argmin(history.history["val_loss"]))
        # else:
        #     # The best epoch is the one with the highest validation loss
        #     best_idx = int(np.argmax(history.history["val_loss"]))
            
        # Choose best epoch based on validation accuracy
        best_idx = int(np.argmax(history.history["val_accuracy"]))

        best_train_loss = float(history.history["loss"][best_idx])
        best_val_loss = float(history.history["val_loss"][best_idx])
        best_train_acc = float(history.history["accuracy"][best_idx])
        best_val_acc = float(history.history["val_accuracy"][best_idx])

        trial.set_user_attr("best_epoch", best_idx + 1)
        trial.set_user_attr("best_train_loss", best_train_loss)
        trial.set_user_attr("best_val_loss", best_val_loss)
        trial.set_user_attr("s008_loss", float(s008_loss))
        trial.set_user_attr("s009_loss", float(s009_loss))

        trial.set_user_attr("best_train_accuracy", best_train_acc)
        trial.set_user_attr("best_val_accuracy", best_val_acc)
        trial.set_user_attr("s008_accuracy", float(s008_acc))
        trial.set_user_attr("s009_accuracy", float(s009_acc))

        # ——————————————————————————————— Save history ——————————————————————————————— #
        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        # Create a DataFrame with all history data
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
            "train_accuracy": history.history["accuracy"],
            "val_accuracy": history.history["val_accuracy"],
        }

        # Convert to DataFrame and save as CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        # ————————————————————————— Finish the current trial ————————————————————————— #
        if len(history.history["val_loss"]) > 1:  # Termination Judgement Report
            report_cross_validation_scores(trial, scores=history.history["val_loss"])

        return best_val_loss  # Value to minimize or maximize
    except ValueError as e:
        # Catch invalid model configurations
        # e.g., when a pooling operation results in negative dimension size
        if "Negative dimension size" in str(e):
            raise optuna.TrialPruned("Pruned, invalid pooling config") from e
        raise
    except Exception as e:
        log_trial_error(
            trial=trial,
            exc=e,
            logs_dir=logs_dir,
            prune_on={
                tf.errors.ResourceExhaustedError: None,
                tf.errors.InternalError: None,
                tf.errors.UnavailableError: None,
            },
            propagate={
                optuna.exceptions.TrialPruned: None,
            },
            force_crash_oom=None,  # Crash after X occurrences of OOM
        )

## Main

In [ ]:
# # Search space:
# base_path = f"{RUN_DIR}/search_space/"
# (
#     x_s008_lidar_train,
#     x_s008_lidar_val,
#     x_s008_coord_train,
#     x_s008_coord_val,
#     y_s008_train,
#     y_s008_val,
# ) = train_test_split(
#     s008_lidar_input,
#     s008_coord_input,
#     s008_y_train,
#     test_size=0.2,
#     random_state=0,
#     shuffle=True,
# )


# plot_model_param_distribution(
#     lambda trial: build_model(
#         trial=trial,
#         kparams=kparams,
#         show_summary=False,
#         train_seed=0,
#     ),
#     benchmark_training=False,
#     fit_x=(x_s008_lidar_train, x_s008_coord_train),
#     fit_y=y_s008_train,
#     fit_validation_data=((x_s008_lidar_val, x_s008_coord_val), y_s008_val),
#     bytes_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
#     batch_size=1,
#     n_trials=NUM_TRIALS,
#     fig_save_path=f"{base_path}model_param_distribution.png",
#     csv_path=f"{base_path}model_param_distribution.csv",
#     logs_dir=f"{base_path}logs/",
#     corr_csv_path=f"{base_path}model_param_distribution_corr.csv",
#     # plot_model_dir=f"{base_path}plots/",
#     figsize=(18, 6),
# )

In [ ]:
study = run_study(
    objective=objective,
    run_dir=RUN_DIR,
    num_trials=NUM_TRIALS,
    sampler_seed=SAMPLER_SEED,
    direction=DIRECTION,
    top_k=TOP_K,
    rank_key=RANK_KEY,
    order=ORDER,
    convergence_epoch_column="train_loss",
    convergence_epoch_direction="minimize",
    init_study_dirs=[
        "args",
        "fig",
        "backup",
        "history",
        "scaler",
        "model",
        "logs",
        "tensorboard",
    ],
    cleanup_paths=[
        ("model", "trial_{trial_id}.keras"),
        ("fig", "trial_{trial_id}.png"),
        ("history", "trial_{trial_id}.csv"),
        ("tensorboard", "trial_{trial_id}"),
        ("scaler", "trial_{trial_id}.pkl"),
    ],
    rename_paths=[
        ("model", ".keras"),
        ("fig", ".png"),
        ("history", ".csv"),
        ("scaler", ".pkl"),
    ],
    extra_attrs=[
        "best_epoch",
        "best_train_loss",
        "best_val_loss",
        "s008_loss",
        "s009_loss",
        "best_train_accuracy",
        "best_val_accuracy",
        "s008_accuracy",
        "s009_accuracy",
    ],
    variance_threshold=1e-12,
    prune_threshold=50,
    patience=100,
)